## Interpreting pretrained Timer with WinTSR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/pretrained_timer.ipynb)

Use [Timer](https://arxiv.org/abs/2402.02368), a generative pretrained time-series Transformer, to make a real zero-shot forecast and reveal which observations drove it with WinTSR. We use the smallest public [Timer checkpoint on Hugging Face](https://huggingface.co/thuml/timer-base-84m): 84M parameters.

In [ ]:
%pip install -q tslens transformers==4.40.1

## 1. Load Timer and make a zero-shot forecast

ETTh2 records electricity-transformer temperatures every hour. We give Timer 1,440 oil-temperature observations and ask it for the next 96, with no training or fine-tuning.

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM

DATA_URL = "https://raw.githubusercontent.com/WenWeiTHU/TimeSeriesDatasets/refs/heads/main/ETT-small/ETTh2.csv"
LOOKBACK, PRED_LEN = 1440, 96
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

series = torch.tensor(pd.read_csv(DATA_URL)["OT"].dropna().to_numpy(), dtype=torch.float32)
starts = (0, 24)
x = torch.stack([series[s : s + LOOKBACK] for s in starts]).unsqueeze(-1).to(device)
ground_truth = series[LOOKBACK : LOOKBACK + PRED_LEN]

model = AutoModelForCausalLM.from_pretrained(
    "thuml/timer-base-84m", trust_remote_code=True
).to(device).eval()
with torch.inference_mode():
    output = model.generate(x[0, :, 0].unsqueeze(0), max_new_tokens=PRED_LEN)
prediction = output[0, -PRED_LEN:].detach().cpu()
print(f"device: {device} | context: {tuple(x[0, :, 0].shape)} | forecast: {tuple(prediction.shape)}")

In [ ]:
import matplotlib.pyplot as plt

context = x[0, :, 0].detach().cpu()
forecast_steps = torch.arange(LOOKBACK, LOOKBACK + PRED_LEN)
plt.figure(figsize=(12, 4))
plt.plot(torch.arange(LOOKBACK), context, label="lookback", linewidth=1)
plt.plot(forecast_steps, ground_truth, label="ground truth", linewidth=2)
plt.plot(forecast_steps, prediction, label="Timer forecast", linewidth=2)
plt.axvline(LOOKBACK - 1, color="0.4", linestyle="--", linewidth=1)
plt.xlabel("hour")
plt.ylabel("oil temperature (OT)")
plt.title("Timer zero-shot forecast on ETTh2")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Adapt `generate()` to WinTSR

WinTSR expects a callable from `(batch, seq_len, n_features)` to predictions. Timer's `generate()` instead consumes a 2D sequence and runs an autoregressive loop, so a small adapter makes those shapes explicit and lets baseline replacements flow into generation. `generate()` exposes no gradients; this integration therefore works with WinTSR's perturbation-based default backend, not a gradient attribution method.

In [ ]:
class TimerWrapper(torch.nn.Module):
    def __init__(self, model, pred_len):
        super().__init__()
        self.model = model
        self.pred_len = pred_len

    def forward(self, x):
        seqs = x.squeeze(-1)
        out = self.model.generate(seqs, max_new_tokens=self.pred_len)
        return out[:, -self.pred_len:].unsqueeze(-1)

wrapped_model = TimerWrapper(model, PRED_LEN).eval()
with torch.inference_mode():
    print("adapter output:", tuple(wrapped_model(x[:, -512:, :]).shape))

## 3. Attribute the recent context

A full 1,440-step context for every perturbation is slow. Two nearby samples and their most recent 512 steps keep both WinTSR stages practical; `threshold=0.5` skips the lower half of time steps during stage two.

In [ ]:
from tslens import WinTSR

inputs = x[:, -512:, :]
baselines = torch.zeros_like(inputs)
attr = WinTSR(wrapped_model).attribute(
    inputs=inputs,
    baselines=baselines,
    threshold=0.5,
    show_progress=True,
)
print("attributions:", tuple(attr.shape), "= (batch, horizon, time, feature)")

## 4. See which time steps mattered

The single-feature heatmap becomes an attribution-intensity trace. We average absolute attribution over all 96 forecast horizons for sample 0.

In [ ]:
recent = inputs[0, :, 0].detach().cpu()
saliency = attr[0].abs().mean(dim=0).squeeze(-1).detach().cpu()
steps = torch.arange(-len(recent), 0)

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True, height_ratios=(2, 1))
axes[0].plot(steps, recent, color="tab:blue", linewidth=1)
axes[0].set_ylabel("OT")
axes[0].set_title("Timer input and WinTSR attribution")
axes[0].grid(alpha=0.25)
axes[1].fill_between(steps, saliency, color="tab:orange", alpha=0.8)
axes[1].plot(steps, saliency, color="tab:orange", linewidth=0.8)
axes[1].set_xlabel("hours before forecast")
axes[1].set_ylabel("attribution")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Read the pattern, not a story

Look for concentration near the forecast boundary (a recency bias) and spikes around repeating daily or weekly lags. Those are hypotheses to check against the plot, not guaranteed properties of this sample: attribution reports sensitivity to these baseline replacements, and a zero baseline may itself be far from typical oil temperatures.

## Next steps

- Try a seasonal or local-mean baseline and compare whether the important lags persist.
- Index `attr[:, h]` instead of averaging to explain one forecast horizon.
- Increase the context toward Timer's 2,880-step maximum on a GPU, or explore [Timer-XL](https://arxiv.org/abs/2410.04803) for long-context and multivariate forecasting.

If this was useful, please star the [repository](https://github.com/khairulislam/tslens). Please cite the following if you use our work:

```bibtex
@article{liu2024timer,
  title={Timer: Generative Pre-trained Transformers Are Large Time Series Models},
  author={Liu, Yong and Zhang, Haoran and Li, Chenyu and Huang, Xiangdong and Wang, Jianmin and Long, Mingsheng},
  journal={arXiv preprint arXiv:2402.02368},
  year={2024}
}

@article{liu2024timerxl,
  title={Timer-XL: Long-Context Transformers for Unified Time Series Forecasting},
  author={Liu, Yong and Qin, Guo and Huang, Xiangdong and Wang, Jianmin and Long, Mingsheng},
  journal={arXiv preprint arXiv:2410.04803},
  year={2024}
}

@article{islam2024wintsr,
  title={WinTSR: A Windowed Temporal Saliency Rescaling Method for Interpreting Time Series Deep Learning Models},
  author={Islam, Md Khairul and Fox, Judy},
  journal={arXiv preprint arXiv:2412.04532},
  year={2024}
}
```